[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-06-compound-charts.ipynb#scrollTo=aa1bb2cc)

---
# Day 6 · Compound Charts
**certified-journeys / altair-certified** · Day 6 · Compound Charts

> **Goal for today:** Compose multi-mark and multi-panel Altair charts using layering (`+`), horizontal/vertical concatenation (`|` / `&`), faceting, and the `repeat` operator for matrix plots.

In [ ]:
%pip install -q altair vega-datasets

## Step 1 · Layering marks with the `+` operator

The `+` operator in Altair creates a **layer chart** — multiple mark types drawn on the **same coordinate space and the same axes**.

```python
bar_chart + line_chart   # line overlaid on bars, shared x/y axes
```

Layering is the right choice when:
- Marks share a **meaningful axis** (e.g. bar heights and a trend line both represent the same quantity)
- You want to annotate a chart (reference lines, text labels on top of bars)

Use `alt.layer(a, b, c)` as the functional equivalent of `a + b + c`.

In [ ]:
import altair as alt
from vega_datasets import data
import pandas as pd

cars = data.cars()

# Base chart shared by both layers
base = alt.Chart(cars)

# Layer 1: bars showing mean horsepower per origin
bars = base.mark_bar(opacity=0.7).encode(
    x=alt.X('Origin:N', title='Country of Origin'),
    y=alt.Y('mean(Horsepower):Q', title='Mean Horsepower'),
    color=alt.Color('Origin:N', legend=None)
)

# Layer 2: text labels on top of each bar
labels = base.mark_text(dy=-8, fontSize=12, fontWeight='bold').encode(
    x=alt.X('Origin:N'),
    y=alt.Y('mean(Horsepower):Q'),
    text=alt.Text('mean(Horsepower):Q', format='.0f')  # format the label
)

# Combine with + operator (layering)
(bars + labels).properties(title='Mean Horsepower by Origin', width=300)

### What just happened?

- `bars + labels` creates a layer chart — both marks share the same X and Y axes.
- The `text` mark uses the same `mean(Horsepower):Q` encoding, so labels are positioned at the bar tops.
- **`dy=-8`** nudges the label 8 pixels upward so it sits above the bar tip rather than inside it.
- Layers are drawn in order — later layers appear on top.

## Step 2 · resolve_scale — independent y-axes across layers

By default, layers **share the same scale** on every channel. When two layers have different units or magnitudes (e.g. revenue in millions and count in thousands), you want independent Y axes.

`resolve_scale(y='independent')` gives each layer its own Y axis — Altair renders the second axis on the right side.

In [ ]:
seattle = data.seattle_weather()

# Layer 1: precipitation as bars (left Y axis)
precip_bar = alt.Chart(seattle).mark_bar(color='steelblue', opacity=0.5).encode(
    x=alt.X('yearmonth(date):T', title='Month'),
    y=alt.Y('sum(precipitation):Q', title='Total Precipitation (mm)')
)

# Layer 2: mean max temperature as line (right Y axis — independent)
temp_line = alt.Chart(seattle).mark_line(color='firebrick', strokeWidth=2).encode(
    x=alt.X('yearmonth(date):T'),
    y=alt.Y('mean(temp_max):Q', title='Avg Max Temp (°C)')
)

# resolve_scale makes each layer's Y axis independent
alt.layer(precip_bar, temp_line).resolve_scale(
    y='independent'   # two separate Y axes: left (precipitation), right (temp)
).properties(
    title='Seattle: Precipitation vs Temperature',
    width=500
)

### What just happened?

- `.resolve_scale(y='independent')` tells Vega-Lite to give each layer its own Y scale.
- The second layer's Y axis appears on the **right side** of the chart automatically.
- **Use independent scales with care** — dual-axis charts can mislead if the scales are chosen to imply spurious correlation. Always label both axes clearly.

## Step 3 · Horizontal and vertical concatenation with `|` and `&`

Concatenation places charts in **separate panels with independent axes and scales**:

```python
chart_a | chart_b          # side by side (hconcat)
chart_a & chart_b          # stacked vertically (vconcat)
(chart_a | chart_b) & chart_c  # 2-column top row, single bottom row
```

Use concatenation when charts are **independent views** of different variables — they don't share a meaningful axis.

In [ ]:
# Three scatter plots concatenated horizontally, one per Origin
def scatter_for_origin(origin_name):
    """Return a scatterplot filtered to a single car origin."""
    return alt.Chart(cars).mark_point(opacity=0.8).encode(
        x=alt.X('Horsepower:Q', scale=alt.Scale(domain=[0, 250])),
        y=alt.Y('Miles_per_Gallon:Q', scale=alt.Scale(domain=[0, 50])),
        tooltip=['Name:N', 'Horsepower:Q', 'Miles_per_Gallon:Q']
    ).transform_filter(
        alt.datum.Origin == origin_name
    ).properties(title=origin_name, width=200, height=180)

# Concatenate three independent panels side by side
scatter_for_origin('USA') | scatter_for_origin('Europe') | scatter_for_origin('Japan')

### What just happened?

- `|` produces three completely independent panels — each has its own axes.
- We pinned both scales to the same `domain` so comparisons are visually fair — this is a **manual** step; concatenation does not auto-align scales unless you use `resolve`.
- **`+` vs `|`**: use `+` when marks share axes (layering), `|` when views are independent (concatenation).

## Step 4 · Small multiples with `facet`

`facet` generates a panel for each unique value of a field — similar to `|` but **automatic and data-driven**. Every panel has the same chart spec, making it the canonical "small multiples" pattern.

Key parameters:

| Parameter | Effect |
|-----------|--------|
| `facet='field:N'` | One panel per value of the field |
| `columns=N` | Wrap into N-column grid |
| `resolve` | Share or isolate axes/scales across facets |

In [ ]:
# Small multiples: HP distribution faceted by Origin (2-column grid)
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=15), title='Horsepower'),
    y=alt.Y('count():Q', title='Count')
).facet(
    facet='Origin:N',   # one panel per origin
    columns=3           # arrange in a row of 3
).properties(
    title='Horsepower Distribution by Origin (small multiples)'
)

### What just happened?

- `.facet(facet='Origin:N', columns=3)` automatically generates one panel per unique Origin value.
- By default, facets **share X and Y scales** — this makes comparisons across panels meaningful.
- Use `resolve_scale(x='independent', y='independent')` after `.facet(...)` to give each panel its own scale when distributions differ dramatically in range.

## Step 5 · Shared vs independent axes across facets

`resolve` controls whether facet panels share or independently compute their scale domains.

```python
.facet(...).resolve_scale(y='independent')  # each panel autoscales its own Y
```

| `resolve_scale(axis=...)` | When to use |
|---------------------------|-------------|
| `'shared'` (default) | Comparing magnitudes across facets |
| `'independent'` | When ranges differ so much that shared scale compresses the data |

Independent axes help **reveal shape** but **hide magnitude comparisons** — choose deliberately.

In [ ]:
# Compare shared vs independent Y scale across facets
chart_shared = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=10)),
    y='count():Q'
).facet('Origin:N', columns=3).properties(title='Shared Y scale')

chart_independent = alt.Chart(cars).mark_bar().encode(
    x=alt.X('Horsepower:Q', bin=alt.Bin(maxbins=10)),
    y='count():Q'
).facet('Origin:N', columns=3).resolve_scale(
    y='independent'   # each panel autoscales its own Y
).properties(title='Independent Y scale')

chart_shared & chart_independent  # vertical concat for comparison

### What just happened?

- **Shared Y** (top): Japanese cars look sparse because the USA range dominates the scale.
- **Independent Y** (bottom): each panel fills its own space, revealing shape more clearly, but you can no longer compare bar heights across panels.
- We used `&` (vertical concat) to stack the two faceted charts, letting you see both at once.
- **Rule of thumb:** default to shared scale; switch to independent only when shape matters more than magnitude.

## Step 6 · `repeat` — scatter plot matrix (SPLOM)

`repeat` is a higher-level operator that generates a grid of charts, varying one or two fields systematically. It's the canonical way to build a **scatter plot matrix (SPLOM)** in Altair.

```python
alt.Chart(df).mark_point().encode(
    x=alt.X(alt.repeat('column'), type='quantitative'),
    y=alt.Y(alt.repeat('row'),    type='quantitative'),
).repeat(
    row=['field_a', 'field_b', 'field_c'],
    column=['field_a', 'field_b', 'field_c']
)
```

`alt.repeat('row')` and `alt.repeat('column')` are **placeholders** that Altair replaces with each field from the `repeat()` call.

In [ ]:
# Build a 3×3 SPLOM of key numeric car fields
numeric_fields = ['Horsepower', 'Miles_per_Gallon', 'Weight_in_lbs']

alt.Chart(cars).mark_point(size=25, opacity=0.6).encode(
    x=alt.X(alt.repeat('column'), type='quantitative'),  # repeat placeholder for x
    y=alt.Y(alt.repeat('row'),    type='quantitative'),  # repeat placeholder for y
    color='Origin:N'
).repeat(
    row=numeric_fields,
    column=numeric_fields
).properties(
    title='Scatter Plot Matrix — Cars Dataset'
)

### What just happened?

- `repeat(row=..., column=...)` generated 9 panels (3×3) — one per (row field, column field) pair.
- `alt.repeat('column')` and `alt.repeat('row')` are resolved at render time to the actual field names.
- The diagonal panels (e.g. Horsepower vs Horsepower) show a perfect diagonal line — expected and harmless in a SPLOM.
- **SPLOM advantage over manual `|`:** adding or removing a field requires changing only the `numeric_fields` list, not the chart structure.

In [ ]:
# Challenge: Compound chart — bar + trend overlay with dual axes
#
# Using the 'seattle_weather' dataset:
#   1. Create a bar chart of monthly total precipitation (sum(precipitation))
#      with yearmonth(date) on X
#   2. Create a line chart of monthly mean temp_max
#   3. Layer them with + and give them independent Y scales (resolve_scale)
#   4. Concatenate BELOW that a faceted histogram of precipitation
#      faceted by 'weather' field (sunny/rain/fog/drizzle/snow), columns=5
#   5. Set shared X titles and appropriate Y titles on both layers
#
# Expected layout:
#   [Top row: layered bar+line dual-axis chart, width=500]
#   [Bottom row: faceted histograms by weather type, columns=5]

# Your solution here:
# seattle = data.seattle_weather()

# precip_bars = alt.Chart(seattle).mark_bar(...).encode(
#     x=alt.X('yearmonth(date):T', ...),
#     y=alt.Y('sum(precipitation):Q', title=___)
# )

# temp_line = alt.Chart(seattle).mark_line(...).encode(
#     x=___,
#     y=alt.Y('mean(temp_max):Q', title=___)
# )

# top = alt.layer(precip_bars, temp_line).resolve_scale(y=___).properties(width=500)

# bottom = alt.Chart(seattle).mark_bar().encode(
#     x=alt.X('precipitation:Q', bin=alt.Bin(maxbins=10)),
#     y='count():Q'
# ).facet(facet=___, columns=___)

# top & bottom

---
## Day 6 key concepts recap

| Operator / method | What it does | Shares axes? |
|-------------------|--------------|-------------|
| `chart_a + chart_b` | Layer: same coordinate space, shared axes | Yes |
| `chart_a \| chart_b` | hconcat: side-by-side panels | No |
| `chart_a & chart_b` | vconcat: vertically stacked panels | No |
| `.facet(field, columns=N)` | Data-driven small multiples | Shared by default |
| `.repeat(row=..., column=...)` | Matrix chart / SPLOM | Per-panel by default |
| `.resolve_scale(y='independent')` | Give each layer/facet its own Y scale | No |

> **Tip:** `+` is layering (same coordinate space, same axes), `|` and `&` are concatenation (separate panels with separate axes). Choose layering when marks share a meaningful axis; concatenation when they are independent views.

---
## What's next
**Day 7** → Interactivity — selections, brushing, cross-filtering, and tooltips.

Mark Day 6 complete in your [tracker](../index.html).